# Policies and assessment

**Goal.** Define explicit substitution and linear-transfer policies, inspect
their provenance, and summarize their behavior over several elements.

**Prerequisites.** Familiarity with direct `get_*` and provenance-carrying
`lookup_*` calls from the scalar-data quickstart is helpful. All values in
this notebook are scalar radii in angstrom.

## Setup

Import the public namespace used for policy, dataset, transfer, and
assessment objects.

In [1]:
import atomref as ar


## Ordered substitution

This policy prefers Cordero covalent radii and explicitly substitutes the
legacy CSD covalent table when the preferred table lacks an element. The
result reports that substitution and the dataset that supplied it.

In [2]:
covalent_policy = ar.RadiiPolicy(
    kind='covalent',
    base_set='cordero2008',
    transfers=(
        ar.SubstitutionTransfer(
            source=ar.DatasetRef('covalent_radius', 'csd_legacy_cov')
        ),
    ),
)
lookup = ar.lookup_covalent_radius('Bk', policy=covalent_policy)
print(lookup.source)
print(f"{lookup.value:.12f}")
print(lookup.resolved_from)


transfer_substitution
1.540000000000
(DatasetRef(quantity='covalent_radius', set_id='csd_legacy_cov'),)


## Linear transfer from support data

For a missing van der Waals value, a fitted relation uses the packaged Rahm
atomic-radius support set. The fit record exposes its coefficients and
training-point count rather than hiding the inference.

In [3]:
vdw_policy = ar.RadiiPolicy(
    kind='van_der_waals',
    base_set='alvarez2013',
    transfers=(
        ar.LinearTransfer(
            predictors=(ar.DatasetRef('atomic_radius', 'rahm2016'),)
        ),
    ),
)
lookup = ar.lookup_vdw_radius('Pm', policy=vdw_policy)
print(f"{lookup.value:.12f}")
print(lookup.source)
print(
    f"slope={lookup.fit.coefficients[0]:.12f} intercept={lookup.fit.intercept:.12f} n={lookup.fit.n_points}"
)


2.897226539515
transfer_linear
slope=1.135336645553 intercept=-0.315776167399 n=90


## Assess behavior over a selection

Assessment counts direct, transferred, and missing outcomes and can retain a
per-element result for auditing the policy before it enters an algorithm.

In [4]:
assessment = ar.assess_radii_policy(
    ['C', 'Xe', 'Pm', 'Bk'],
    policy=vdw_policy,
    detail=True,
)
print(assessment.n_base, assessment.n_transfer_linear, assessment.n_missing)
for row in assessment.per_element:
    value = 'None' if row.lookup.value is None else f"{row.lookup.value:.12f}"
    print(row.symbol, row.lookup.source, value)


3 1 0
C base 1.770000000000
Xe base 2.060000000000
Pm transfer_linear 2.897226539515
Bk base 3.400000000000


## What this demonstrated and limitations

Policies make fallback order and inferred values explicit, while assessment
shows how that choice behaves over a requested element set. Transfer quality
still depends on scientifically appropriate target and support datasets.
These scalar policies do not apply to proatomic radial profiles.